# Assignment 13: Titanic Preprocessing and Model Comparison

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

COLUMNS = ["survived", "pclass", "sex", "age", "fare", "embarked"]


def load_titanic():
    full = sns.load_dataset("titanic")
    return full[COLUMNS].copy()


def missing_value_report(df):
    return df.isna().sum()


def impute_missing(df):
    result = df.copy()
    result["age"] = result["age"].fillna(result["age"].mean())
    result["embarked"] = result["embarked"].fillna(result["embarked"].mode()[0])
    return result


def encode_categorical(df):
    result = df.copy()
    encoder = LabelEncoder()
    result["sex"] = encoder.fit_transform(result["sex"])
    result = pd.get_dummies(result, columns=["embarked"])
    dummy_cols = [c for c in result.columns if c.startswith("embarked_")]
    result[dummy_cols] = result[dummy_cols].astype(int)
    return result, encoder


def scale_numeric(df, columns, scaler=None):
    result = df.copy()
    if scaler is None:
        scaler = StandardScaler()
        result[columns] = scaler.fit_transform(result[columns])
    else:
        result[columns] = scaler.transform(result[columns])
    return result, scaler


def split_data(df, target="survived", test_size=0.2, random_state=42):
    X = df.drop(columns=[target])
    y = df[target]
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)

In [2]:
# sample answer: steps 1 to 5 applied in order
raw = load_titanic()
missing_report = missing_value_report(raw)
print(missing_report)

imputed = impute_missing(raw)
encoded, sex_encoder = encode_categorical(imputed)
scaled, scaler = scale_numeric(encoded, ["age", "fare"])
X_train, X_test, y_train, y_test = split_data(scaled)
print(scaled.head())
print(X_train.shape, X_test.shape)

survived      0
pclass        0
sex           0
age         177
fare          0
embarked      2
dtype: int64
   survived  pclass  sex       age      fare  embarked_C  embarked_Q  \
0         0       3    1 -0.592481 -0.502445           0           0   
1         1       1    0  0.638789  0.786845           1           0   
2         1       3    0 -0.284663 -0.488854           0           0   
3         1       1    0  0.407926  0.420730           0           0   
4         0       3    1  0.407926 -0.486337           0           0   

   embarked_S  
0           1  
1           0  
2           1  
3           1  
4           1  
(712, 7) (179, 7)


In [3]:
# edge and negative case checks for steps 1 to 5, kept separate from the analysis above
assert missing_report["age"] > 0
assert missing_report["embarked"] > 0
assert missing_report["survived"] == 0
assert missing_report["fare"] == 0

no_missing_report = missing_value_report(raw.dropna())
assert (no_missing_report == 0).all()

small = pd.DataFrame({"age": [10.0, 20.0, np.nan], "embarked": ["S", "S", np.nan]})
small_imputed = impute_missing(small)
assert small_imputed["age"].isna().sum() == 0
assert small_imputed.loc[2, "age"] == 15.0
assert small_imputed.loc[2, "embarked"] == "S"

try:
    impute_missing(pd.DataFrame({"embarked": ["S", np.nan]}))
    assert False, "expected KeyError for a missing age column"
except KeyError:
    pass

assert imputed["age"].isna().sum() == 0
assert imputed["embarked"].isna().sum() == 0

encoded_check, _ = encode_categorical(imputed)
assert set(encoded_check["sex"].unique()) <= {0, 1}
embarked_dummy_cols = [c for c in encoded_check.columns if c.startswith("embarked_")]
assert len(embarked_dummy_cols) >= 2
assert (encoded_check[embarked_dummy_cols].sum(axis=1) == 1).all()

single_category = pd.DataFrame({"sex": ["male", "female"], "embarked": ["S", "S"]})
single_encoded, _ = encode_categorical(single_category)
assert [c for c in single_encoded.columns if c.startswith("embarked_")] == ["embarked_S"]

try:
    encode_categorical(pd.DataFrame({"sex": ["male", "female"]}))
    assert False, "expected KeyError for a missing embarked column"
except KeyError:
    pass

scaled_check, fit_scaler = scale_numeric(encoded_check, ["age", "fare"])
assert abs(scaled_check["age"].mean()) < 1e-8
assert abs(scaled_check["age"].std(ddof=0) - 1.0) < 1e-8

reused_scaled, _ = scale_numeric(encoded_check.head(10), ["age", "fare"], scaler=fit_scaler)
assert not np.isnan(reused_scaled["age"]).any()

try:
    fit_scaler.transform(encoded_check[["age"]])
    assert False, "expected ValueError for a mismatched column count"
except ValueError:
    pass

assert abs(len(X_test) / len(scaled) - 0.2) < 0.02
train_a, test_a, ytr_a, yte_a = split_data(scaled)
train_b, test_b, ytr_b, yte_b = split_data(scaled)
assert train_a.index.equals(train_b.index)

try:
    split_data(scaled, target="not_a_column")
    assert False, "expected KeyError for an unknown target column"
except KeyError:
    pass

print("all tests passed")

all tests passed


## Preprocessing and model pipeline with cross-validation

In [4]:
# raw split, kept separate from the manually cleaned data used above
raw_train, raw_test, raw_y_train, raw_y_test = train_test_split(
    raw.drop(columns=["survived"]), raw["survived"], test_size=0.2, random_state=42, stratify=raw["survived"]
)

numeric_features = ["age", "fare"]
categorical_features = ["sex", "embarked"]

numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
], remainder="passthrough")

models = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "knn": KNeighborsClassifier(),
}

results = {}
for name, model in models.items():
    pipeline = Pipeline([("preprocess", preprocessor), ("model", model)])
    scores = cross_val_score(pipeline, raw_train, raw_y_train, cv=5, scoring="accuracy")
    results[name] = (scores.mean(), scores.std())

comparison = pd.DataFrame(results, index=["mean_accuracy", "std_accuracy"]).T
print(comparison)

                     mean_accuracy  std_accuracy
logistic_regression       0.797813      0.020863
random_forest             0.800650      0.030981
knn                       0.814695      0.025830


In [5]:
# fit the best performing model on the training set and check test set accuracy
best_name = comparison["mean_accuracy"].idxmax()
best_pipeline = Pipeline([("preprocess", preprocessor), ("model", models[best_name])])
best_pipeline.fit(raw_train, raw_y_train)
test_predictions = best_pipeline.predict(raw_test)
test_accuracy = accuracy_score(raw_y_test, test_predictions)
print("best model:", best_name)
print("test accuracy:", test_accuracy)

best model: knn
test accuracy: 0.8379888268156425


In [6]:
# edge and negative case checks for the pipeline, kept separate from the analysis above
for name, (mean_acc, _) in results.items():
    assert 0.5 < mean_acc < 1.0, name

assert 0.5 < test_accuracy < 1.0

one_row = raw_train.iloc[[0]].copy()
one_row["age"] = np.nan
prediction = best_pipeline.predict(one_row)
assert prediction[0] in (0, 1)

try:
    cross_val_score(best_pipeline, raw_train, raw_y_train, cv=1)
    assert False, "expected ValueError for cv=1"
except ValueError:
    pass

print("all tests passed")

all tests passed
